In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [3]:
BASE_DIR = Path().resolve().parent
PROCESSED_PATH = BASE_DIR / "data" / "processed" / "creditcard_clean.csv"

df = pd.read_csv(PROCESSED_PATH)

In [4]:
df["Amount_log"] = np.log1p(df["Amount"])
df = df.drop(columns=["Amount"])

In [5]:
df['Time'].value_counts()

Time
3767.0      21
3770.0      20
3750.0      19
19912.0     19
3749.0      17
            ..
121280.0     1
11.0         1
13.0         1
14.0         1
15.0         1
Name: count, Length: 124592, dtype: int64

In [6]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Class,Amount_log
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,5.014760
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,1.305626
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,5.939276
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,4.824306
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0,4.262539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283721,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0,0.570980
283722,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,0,3.249987
283723,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,0,4.232366
283724,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,0,2.397895


In [7]:
df["Time_hour"] = (df["Time"] % 86400) / 3600

df["Time_sin"] = np.sin(2 * np.pi * df["Time_hour"] / 24)
df["Time_cos"] = np.cos(2 * np.pi * df["Time_hour"] / 24)

df = df.drop(columns=["Time","Time_hour"])

In [8]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Logistic Regression

In [9]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

print("LOGISTIC REGRESSION")
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("PR AUC:", average_precision_score(y_test, y_prob))
print(confusion_matrix(y_test, y_pred))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56651
           1       0.06      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.55     56746
weighted avg       1.00      0.97      0.99     56746

ROC AUC: 0.9635693707269534
PR AUC: 0.6805680059611355
[[55241  1410]
 [   12    83]]


### Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("RANDOM FOREST")
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("PR AUC:", average_precision_score(y_test, y_prob))
print(confusion_matrix(y_test, y_pred))

RANDOM FOREST
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.99      0.71      0.82        95

    accuracy                           1.00     56746
   macro avg       0.99      0.85      0.91     56746
weighted avg       1.00      1.00      1.00     56746

ROC AUC: 0.918243093214316
PR AUC: 0.8058374406053248
[[56650     1]
 [   28    67]]


### Xgboost

In [11]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("XGBOOST")
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("PR AUC:", average_precision_score(y_test, y_prob))
print(confusion_matrix(y_test, y_pred))

XGBOOST
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.90      0.78      0.84        95

    accuracy                           1.00     56746
   macro avg       0.95      0.89      0.92     56746
weighted avg       1.00      1.00      1.00     56746

ROC AUC: 0.9726970955127843
PR AUC: 0.8224615631900266
[[56643     8]
 [   21    74]]


### LightGBM

In [12]:
lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(X_train, y_train)

y_pred = lgb_model.predict(X_test)
y_prob = lgb_model.predict_proba(X_test)[:, 1]

print("LIGHTGBM")
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("PR AUC:", average_precision_score(y_test, y_prob))
print(confusion_matrix(y_test, y_pred))

LIGHTGBM
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.86      0.77      0.81        95

    accuracy                           1.00     56746
   macro avg       0.93      0.88      0.91     56746
weighted avg       1.00      1.00      1.00     56746

ROC AUC: 0.977405332186267
PR AUC: 0.8185164646096225
[[56639    12]
 [   22    73]]


| Model         | Precision | Recall   | PR AUC   | Verdict                              |
| ------------- | --------- | -------- | -------- | ------------------------------------ |
| Logistic      | 0.06      | **0.87** | 0.68     | catches fraud, too many false alarms |
| Random Forest | **0.99**  | 0.71     | 0.80     | very strict, misses fraud            |
| XGBoost       | 0.90      | **0.78** | **0.82** | best balance                         |
| LightGBM      | 0.86      | 0.77     | 0.81     | very close to XGB                    |


In [14]:
import joblib

ARTIFACTS_DIR = Path().resolve().parent / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

best_model = xgb_model
model_path = ARTIFACTS_DIR / "fraud_model.joblib"

joblib.dump(best_model, model_path)
print("Saved model to:", model_path)

Saved model to: C:\Users\tevin\OneDrive\Desktop\fraud_detection\artifacts\fraud_model.joblib
